# Spectral Generalisation of the Variance Ratio Strategy

## Paper Citation
**Title:** A Spectral Generalisation of the Variance Ratio: Eigenstructure of Long-Horizon Portfolio Covariance and a Multi-Memory Factor Model of U.S. Equity Returns
**Authors:** Anders G Frøseth
**Published:** 2026-07-04
**ArXiv:** [https://arxiv.org/abs/2607.03858](https://arxiv.org/abs/2607.03858)

## Strategy Description
This notebook implements a quantitative trading strategy based on the paper by Anders G Frøseth. The strategy uses a multivariate generalisation of the Lo-MacKinlay variance ratio to decompose long-horizon equity-return dynamics into separate return-channel and volatility-channel memory components. The framework identifies a parsimonious five-factor model that captures persistent, antipersistent, and multi-scale memory in returns and volatility.

The strategy involves:
1. Downloading market data using `yfinance`.
2. Computing factors and features.
3. Generating signals and constructing the portfolio.
4. Performing a vectorized backtest.
5. Calculating performance metrics.
6. Implementing a monitoring stub for live data.


In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT']
PARAMETERS = {
    'lookback': 252,  # 1-year lookback period
    'factor_halflife': 63,  # 3-month halflife for factor decay
}
HYPOTHESIS = "The strategy will capture long-horizon return and volatility dynamics using a five-factor model."

## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start='2010-01-01', end='2023-01-01', group_by='ticker')

# Compute returns
returns = pd.DataFrame()
for ticker in UNIVERSE:
    returns[ticker] = data[ticker]['Adj Close'].pct_change().dropna()

# Compute factors
factors = returns.rolling(window=PARAMETERS['lookback']).mean()
factors = factors.ewm(halflife=PARAMETERS['factor_halflife']).mean()
factors = factors.sub(factors.mean(axis=1), axis=0).div(factors.std(axis=1), axis=0)

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Generate signals
signals = factors.rank(axis=1, pct=True).sub(0.5)

# Position sizing
positions = signals.mul(1 / signals.abs().sum(axis=1), axis=0)

## Phase 4 — Vectorized Backtest

In [ ]:
# Shift signals forward by 1 period to avoid look-ahead bias
signals_shifted = signals.shift(1)

# Calculate portfolio returns
portfolio_returns = (returns * positions).sum(axis=1)

# Calculate cumulative returns
cumulative_returns = (1 + portfolio_returns).cumprod()

## Phase 5 — Performance Metrics

In [ ]:
import scipy.stats as stats

# Calculate performance metrics
annual_return = portfolio_returns.mean() * 252
annual_volatility = portfolio_returns.std() * np.sqrt(252)
sharpe_ratio = annual_return / annual_volatility
sortino_ratio = annual_return / portfolio_returns[portfolio_returns < 0].std() * np.sqrt(252)
max_drawdown = (cumulative_returns / cumulative_returns.cummax() - 1).min()
calmar_ratio = annual_return / (-max_drawdown)

print(f'Annual Return: {annual_return:.2%}')
print(f'Annual Volatility: {annual_volatility:.2%}')
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')

# Plot equity curve
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(cumulative_returns.index, cumulative_returns, label='Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.title('Equity Curve')
plt.legend()
plt.show()

## Phase 6 — Monitoring Stub

In [ ]:
def monitor_daily_pnl(returns, positions):
    daily_pnl = (returns * positions).sum(axis=1)
    current_positions = positions.iloc[-1]
    print(f'Daily P&L: {daily_pnl.iloc[-1]:.2f}')
    print('Current Positions:')
    print(current_positions)

# Example usage
monitor_daily_pnl(returns, positions)